# Baseline of the first ML Model: TF-IDF

                    TEXT
                      │
                      ▼
                TF-IDF
                      │
                      ▼
          Logistic Regression
                      │
                      ▼
                 PREDICTION
                      │
          ┌───────────┼───────────┐
          ▼           ▼           ▼
       Accuracy    Precision      Recall
                                  │
                                  ▼
                                  F1

# Import

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Load the data

Even though we've already worked with the data in the previous notebook, a notebook must be reproducible on its own.

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

dataset = load_dataset("fancyzhx/yelp_polarity")

df_train = dataset["train"].to_pandas()
df_test = dataset["test"].to_pandas()

from src.preprocessing import clean_text

df_train["text"] = df_train["text"].apply(clean_text)
df_test["text"] = df_test["text"].apply(clean_text)

# Create train / validation / test

In [4]:
from sklearn.model_selection import train_test_split

X = df_train["text"]
y = df_train["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_test = df_test["text"]
y_test = df_test["label"]

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Training: 448000
Validation: 112000
Test: 38000


# What if TF-IDF

TF  - Term Frequency               (frequent words)
IDF - Inverse Document Frequency   (rare words)

Review
   │
   ▼
TF-IDF
   │
   ▼
numerical feature vector
   │
   ▼
Logistic Regression

# Build the TF-IDF

Start with a baseline configuration, without finding firstly the max score

In [5]:
tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

- ngram_range=(1, 2) 

Means both unigram and bigram are used.
For example "good" and "not good". They have very different meanings.

- min_df=2 

Ignore terms present in only one document

- max_df=0.95

Ignore terms present in 95% ore more documents

# Text transformation

In [6]:
X_train_tfidf = tfidf.fit_transform(X_train)

X_val_tfidf = tfidf.transform(X_val)

X_test_tfidf = tfidf.transform(X_test)

On the training `fit_transform()` is performed
On the validation/test `transform()` is performed

Never do `fit_transform()` on the validation or test. 

Because the TF-IDF vocabulary must be built solely on the training set.
Otherwise, we would have data leakage.

# Check the matrix

In [ ]:
print("Train shape:", X_train_tfidf.shape)
print("Validation shape:", X_val_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)

# Let's see how sparse the matrix is

In [ ]:
print("Matrix type:", type(X_train_tfidf))
print("Non-zero values:", X_train_tfidf.nnz)

total_values = (
    X_train_tfidf.shape[0] *
    X_train_tfidf.shape[1]
)

sparsity = 1 - (X_train_tfidf.nnz / total_values)

print(f"Sparsity: {sparsity:.4%}")

It ends up with very high sparsity.

That is why the matrix should not be converted into a regular dense NumPy array.

# Build Logistic Regression

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

# Make the first predictions

In [ ]:
y_val_pred = model.predict(X_val_tfidf)
y_test_pred = model.predict(X_test_tfidf)

# Validation evaluation